### General

In [5]:
from pathlib import Path
import os

# Fijar directorio de trabajo al directorio del proyecto
os.chdir(Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent)

# Verificar
print(f"Directorio actual: {os.getcwd()}")
print(f"Archivos en data/: {os.listdir('data')}")

Directorio actual: <project-root>
Archivos en data/: ['links_viviendas.csv']


In [7]:
# ── SECCIÓN 1: IMPORTS ──────────────────────────────
import undetected_chromedriver as uc
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
import re
import os
from datetime import datetime

# ── SECCIÓN 2: CONFIGURACIÓN CENTRAL ────────────────
SOURCES = {
    "idealista":  "idealista",
    "fotocasa":   "fotocasa",
    "pisos":      "pisos.com",
    "habitaclia": "habitaclia",
    "tecnocasa":  "tecnocasa",
    "yaencontre": "yaencontre",
}

# ── TIMING — comportamiento humano ──────────────────
WAIT_PAGE_LOAD  = (5, 9)     # rango carga de página
WAIT_MIN        = 4          # mínimo entre requests
WAIT_MAX        = 15         # máximo entre requests
WAIT_LONG       = (25, 45)   # rango pausa larga
WAIT_LONG_EVERY = (8, 12)    # cada cuántos requests pausa larga

def esperar_entre_requests():
    """Pausa con comportamiento humano"""
    espera = random.uniform(WAIT_MIN, WAIT_MAX)
    # 10% probabilidad de pausa extra — como si te distrajeras
    if random.random() < 0.1:
        espera += random.uniform(15, 30)
    time.sleep(espera)

def esperar_carga_pagina():
    """Espera variable para carga de página"""
    time.sleep(random.uniform(*WAIT_PAGE_LOAD))

def esperar_pausa_larga():
    """Pausa larga entre bloques de requests"""
    time.sleep(random.uniform(*WAIT_LONG))

# ── SECCIÓN 3: CARGAR LINKS ─────────────────────────
df = pd.read_csv("data/links_viviendas.csv", header=0)
df.columns = ["url", "extra"]
df = df[["url"]].dropna()
df["url"] = df["url"].str.strip()

def detectar_plataforma(url):
    for nombre, dominio in SOURCES.items():
        if dominio in url:
            return nombre
    return "otro"

df["plataforma"] = df["url"].apply(detectar_plataforma)
df = df.drop_duplicates(subset="url", keep="first")

# Generar sublistas automáticamente
listas = {nombre: df[df["plataforma"] == nombre]["url"].tolist()
          for nombre in SOURCES}

# Reporte
print(f"Total sin duplicados: {len(df)}")
for nombre, lista in listas.items():
    print(f"  {nombre:12}: {len(lista)} links")

# ── SECCIÓN 4: ABRIR NAVEGADOR ──────────────────────
driver = uc.Chrome()
time.sleep(3)
timestamp = datetime.now().strftime("%Y%m%d_%H%M")
print(f"\n✅ Driver listo — {timestamp}")

Total sin duplicados: 73
  idealista   : 44 links
  fotocasa    : 7 links
  pisos       : 4 links
  habitaclia  : 4 links
  tecnocasa   : 2 links
  yaencontre  : 12 links

✅ Driver listo — 20260520_1131


### Idealista

In [8]:
def scrape_idealista(driver, url):
    driver.get(url)
    esperar_carga_pagina()

    soup = BeautifulSoup(driver.page_source, "html.parser")

    texto_pagina = soup.get_text().lower()
    if "ya no está publicado" in texto_pagina:
        return {
            "url": url, "plataforma": "idealista",
            "estado_anuncio": "dado de baja",
            "titulo": None, "ubicacion": None, "precio": None,
            "m2": None, "habitaciones": None, "baños": None,
            "planta": None, "ascensor": None, "tipo": None,
            "estado": None, "año": None, "anunciante": None,
            "comentario": None
        }

    try:
        precio = int(soup.find("span", class_="info-data-price").find("span", class_="txt-bold").text.strip().replace(".", ""))
    except:
        precio = None

    try:
        titulo = soup.find("span", class_="main-info__title-main").text.strip()
    except:
        titulo = None

    try:
        ubicacion = soup.find("span", class_="main-info__title-minor").text.strip()
    except:
        ubicacion = None

    try:
        features = soup.find_all("div", class_="info-features")
        raw = features[0].text.strip() if features else ""
        partes = [p.strip() for p in raw.split("\n") if p.strip()]
    except:
        partes = []

    m2, habitaciones, planta, ascensor = None, None, None, None
    for parte in partes:
        if "m²" in parte:
            m2 = int(re.search(r'\d+', parte).group())
        elif "hab" in parte:
            habitaciones = int(re.search(r'\d+', parte).group())
        elif "lanta" in parte:
            planta = parte
            ascensor = "No" if "sin ascensor" in parte.lower() else "Sí"

    try:
        caract = soup.find("div", class_="details-property-feature-one")
        caract_items = [li.text.strip() for li in caract.find_all("li")]
    except:
        caract_items = []

    año, estado, baños = None, None, None
    for item in caract_items:
        if "construido en" in item.lower():
            año = int(re.search(r'\d{4}', item).group())
        elif "mano" in item.lower() or "estado" in item.lower():
            estado = item
        elif "baño" in item.lower():
            baños = int(re.search(r'\d+', item).group())

    tipo = "Casa" if titulo and any(k in titulo.lower() for k in [
        "casa", "chalet", "villa", "adosada", "mata", "unifamiliar"
    ]) else "Piso"

    try:
        anunciante = soup.find("div", class_="professional-name").text.strip().replace("Profesional", "").strip()
    except:
        anunciante = None

    try:
        comentario = soup.find("div", class_="comment").text.strip()
        comentario = re.sub(r'\s*Leer comentario completo\s*', '', comentario).strip()
    except:
        comentario = None

    return {
        "url": url, "plataforma": "idealista",
        "estado_anuncio": "activo",
        "titulo": titulo, "ubicacion": ubicacion, "precio": precio,
        "m2": m2, "habitaciones": habitaciones, "baños": baños,
        "planta": planta, "ascensor": ascensor, "tipo": tipo,
        "estado": estado, "año": año, "anunciante": anunciante,
        "comentario": comentario
    }

# ── LOOP IDEALISTA ───────────────────────────────────
resultados_idealista = []
for i, url in enumerate(listas["idealista"]):
    print(f"\nScraping {i+1}/{len(listas['idealista'])}: {url}")
    try:
        piso = scrape_idealista(driver, url)
        resultados_idealista.append(piso)
        print(f"✓ {piso['estado_anuncio']} — {piso['titulo']} — {piso['precio']}€")
    except Exception as e:
        print(f"✗ Error: {e}")
    esperar_entre_requests()
    if (i + 1) % random.randint(*WAIT_LONG_EVERY) == 0:
        print("Pausa larga...")
        esperar_pausa_larga()

os.makedirs("data", exist_ok=True)
df_idealista = pd.DataFrame(resultados_idealista)
df_idealista.to_csv(f"data/idealista_scraped_{timestamp}.csv", index=False)
print(f"\n✅ Idealista — {len(resultados_idealista)} pisos")
print(df_idealista["estado_anuncio"].value_counts())


Scraping 1/44: https://www.idealista.com/inmueble/106811791/?utm_medium=socialmedia&utm_campaign=private_sendadtofriend&utm_source=notifications
✓ activo — None — None€

Scraping 2/44: https://www.idealista.com/inmueble/108128775/?utm_medium=socialmedia&utm_campaign=private_sendadtofriend&utm_source=notifications
✓ activo — Piso en venta en Olletas - Sierra Blanquilla — 170000€

Scraping 3/44: https://www.idealista.com/inmueble/108445895/?utm_medium=socialmedia&utm_campaign=private_sendadtofriend&utm_source=notifications
✓ activo — Piso en venta en Manantiales - Estación de Autobuses — 187000€

Scraping 4/44: https://www.idealista.com/inmueble/108668045/?utm_medium=socialmedia&utm_campaign=private_sendadtofriend&utm_source=notifications
✓ activo — Piso en venta en Calle Gran Cardenal, 3 — 199000€

Scraping 5/44: https://www.idealista.com/inmueble/108949215/?utm_medium=socialmedia&utm_campaign=private_sendadtofriend&utm_source=notifications
✓ activo — Chalet pareado en venta en El Moli

### Fotocasa

In [9]:
def scrape_fotocasa(driver, url):
    driver.get(url)
    esperar_carga_pagina()

    soup = BeautifulSoup(driver.page_source, "html.parser")

    texto_pagina = soup.get_text().lower()
    if "this listing is no longer available" in texto_pagina or "ya no está disponible" in texto_pagina:
        return {
            "url": url, "plataforma": "fotocasa",
            "estado_anuncio": "dado de baja",
            "titulo": None, "ubicacion": None, "precio": None,
            "m2": None, "habitaciones": None, "baños": None,
            "planta": None, "ascensor": None, "tipo": None,
            "estado": None, "año": None, "anunciante": None,
            "comentario": None
        }

    try:
        precio_text = soup.find("span", class_="re-DetailHeader-price").text.strip()
        precio = int(precio_text.replace(".", "").replace("€", "").strip())
    except:
        precio = None

    try:
        titulo = soup.find("h1", class_="re-DetailHeader-propertyTitle").text.strip()
    except:
        titulo = None

    try:
        comentario = soup.find("p", class_="re-DetailDescription").text.strip()
    except:
        comentario = None

    try:
        features_text = soup.find("div", class_="re-ContentDetail-featuresListWrapper").text.strip()
    except:
        features_text = ""

    try:
        header_text = soup.find("div", class_="re-ContentDetail-topContainer--main").text.strip()
    except:
        header_text = ""

    m2_match = re.search(r'(\d+)\s*sqm', header_text)
    m2 = int(m2_match.group(1)) if m2_match else None
    if not m2 and comentario:
        m2_fallback = re.search(r'(\d+)\s*m²', comentario)
        if m2_fallback:
            m2 = int(m2_fallback.group(1))

    hab_match = re.search(r'(\d+)\s*bdrm', header_text)
    habitaciones = int(hab_match.group(1)) if hab_match else None
    if not habitaciones and comentario:
        hab_fallback = re.search(r'(\d+)\s*dormitor', comentario.lower()) or re.search(r'(\d+)\s*habitaci', comentario.lower())
        if hab_fallback:
            habitaciones = int(hab_fallback.group(1))

    ban_match = re.search(r'(\d+)\s*bath', header_text)
    baños = int(ban_match.group(1)) if ban_match else None

    planta_match = re.search(r'(\d+(?:st|nd|rd|th)?\s*[Ff]loor)', header_text)
    planta = planta_match.group(1) if planta_match else None

    ascensor = "Sí" if "LiftYes" in features_text else "No" if "LiftNo" in features_text else None

    estado = None
    if "Good" in features_text: estado = "Buen estado"
    elif "New" in features_text: estado = "Nuevo"
    elif "Renovated" in features_text: estado = "Reformado"

    age_match = re.search(r'Age(\d+)\s*to\s*(\d+)\s*years', features_text)
    año = f"{age_match.group(1)}-{age_match.group(2)} años" if age_match else None

    tipo = "Casa" if titulo and any(k in titulo.lower() for k in [
        "house", "chalet", "villa", "adosada", "mata", "unifamiliar", "casa"
    ]) else "Piso"

    try:
        if "in " in titulo: ubicacion = titulo.split("in ")[-1]
        elif " en " in titulo: ubicacion = titulo.split(" en ")[-1]
        else: ubicacion = titulo
    except:
        ubicacion = titulo

    try:
        anunciante = soup.find("p", class_="re-ContactDetail-name").text.strip()
    except:
        anunciante = None

    return {
        "url": url, "plataforma": "fotocasa",
        "estado_anuncio": "activo",
        "titulo": titulo, "ubicacion": ubicacion, "precio": precio,
        "m2": m2, "habitaciones": habitaciones, "baños": baños,
        "planta": planta, "ascensor": ascensor, "tipo": tipo,
        "estado": estado, "año": año, "anunciante": anunciante,
        "comentario": comentario
    }

# ── LOOP FOTOCASA ────────────────────────────────────
resultados_fotocasa = []
for i, url in enumerate(listas["fotocasa"]):
    print(f"\nScraping {i+1}/{len(listas['fotocasa'])}: {url}")
    try:
        piso = scrape_fotocasa(driver, url)
        resultados_fotocasa.append(piso)
        print(f"✓ {piso['estado_anuncio']} — {piso['ubicacion']} — {piso['precio']}€")
    except Exception as e:
        print(f"✗ Error: {e}")
    esperar_entre_requests()
    if (i + 1) % random.randint(*WAIT_LONG_EVERY) == 0:
        print("Pausa larga...")
        esperar_pausa_larga()

df_fotocasa = pd.DataFrame(resultados_fotocasa)
df_fotocasa.to_csv(f"data/fotocasa_scraped_{timestamp}.csv", index=False)
print(f"\n✅ Fotocasa — {len(resultados_fotocasa)} pisos")
print(df_fotocasa["estado_anuncio"].value_counts())


Scraping 1/7: https://www.fotocasa.es/en/buy/home/malaga-capital/air-conditioning-heating-terrace-lift-not-furnished/189631313/d?stc=dis-sharead-sharead&utm_medium=social-share&utm_source=sharead&utm_campaign=sharead
✓ activo — Calle María, Olletas - Sierra Blanquilla — 185000€

Scraping 2/7: https://www.fotocasa.es/en/buy/home/malaga-capital/air-conditioning/189689819/d?stc=dis-sharead-sharead&utm_medium=social-share&utm_source=sharead&utm_campaign=sharead
✓ activo — Avenida de la Paloma, Girón - Las Delicias — 215000€

Scraping 3/7: https://www.fotocasa.es/en/buy/home/malaga-capital/dos-hermanas-nuevo-san-andres/189467628/d?stc=dis-sharead-sharead&utm_medium=social-share&utm_source=sharead&utm_campaign=sharead
✓ activo — Calle Concejal Masso Roura, 4, Dos Hermanas - Nuevo San Andrés — 175000€

Scraping 4/7: https://www.fotocasa.es/en/buy/home/malaga-capital/heating/188993693/d?stc=dis-sharead-sharead&utm_medium=social-share&utm_source=sharead&utm_campaign=sharead
✓ activo — C. Pache

### Pisos.com

In [10]:
def scrape_pisos(driver, url):
    if "alquilar" in url.lower():
        return {
            "url": url, "plataforma": "pisos",
            "estado_anuncio": "alquiler — descartado",
            "titulo": None, "ubicacion": None, "precio": None,
            "m2": None, "habitaciones": None, "baños": None,
            "planta": None, "ascensor": None, "tipo": None,
            "estado": None, "año": None, "anunciante": None,
            "comentario": None
        }

    driver.get(url)
    esperar_carga_pagina()
    soup = BeautifulSoup(driver.page_source, "html.parser")

    texto_pagina = soup.get_text().lower()
    if "anuncio no disponible" in texto_pagina or "ya no está disponible" in texto_pagina:
        return {
            "url": url, "plataforma": "pisos",
            "estado_anuncio": "dado de baja",
            "titulo": None, "ubicacion": None, "precio": None,
            "m2": None, "habitaciones": None, "baños": None,
            "planta": None, "ascensor": None, "tipo": None,
            "estado": None, "año": None, "anunciante": None,
            "comentario": None
        }

    try:
        precio_text = soup.find("div", class_="price__value").text.strip()
        precio = int(re.search(r'[\d\.]+', precio_text).group().replace(".", ""))
    except:
        precio = None

    try:
        titulo = soup.find("h1").text.strip()
    except:
        titulo = None

    try:
        items = soup.find("ul", class_="features-summary").find_all("li", class_="features-summary__item")
        features_texts = [item.text.strip() for item in items]
    except:
        features_texts = []

    m2, habitaciones, baños, planta, ascensor = None, None, None, None, None
    for feat in features_texts:
        feat_lower = feat.lower()
        if "m²" in feat:
            m2_match = re.search(r'(\d+)\s*m²', feat)
            if m2_match: m2 = int(m2_match.group(1))
        elif "habitaci" in feat_lower or "dormitor" in feat_lower:
            hab_match = re.search(r'(\d+)', feat)
            if hab_match: habitaciones = int(hab_match.group(1))
        elif "baño" in feat_lower:
            ban_match = re.search(r'(\d+)', feat)
            if ban_match: baños = int(ban_match.group(1))
        elif "planta" in feat_lower:
            planta = feat
        elif "ascensor" in feat_lower:
            ascensor = "No" if "sin ascensor" in feat_lower else "Sí"

    tipo = "Casa" if titulo and any(k in titulo.lower() for k in ["casa", "chalet", "villa"]) else "Piso"

    try:
        ubicacion = soup.find("span", class_="show-map__address").text.strip()
    except:
        ubicacion = titulo.split(" en ")[-1] if titulo and " en " in titulo else titulo

    try:
        anunciante = soup.find("div", class_="advertiser__name").text.strip()
    except:
        anunciante = None

    try:
        comentario_raw = soup.find("div", class_="description-modal__text") or soup.find("div", class_="js-description")
        comentario = comentario_raw.text.strip()
        if "Traducciones disponibles" in comentario:
            comentario = re.sub(r'Traducciones disponibles.*?Français', '', comentario, flags=re.DOTALL).strip()
            comentario = comentario.split("Mostrar más")[0].strip()
    except:
        comentario = None

    estado, año = None, None
    try:
        for item in soup.find_all("span", class_="features__value"):
            texto = item.text.strip().lower()
            if "buen estado" in texto or "reformado" in texto or "nuevo" in texto:
                estado = item.text.strip()
            año_match = re.search(r'\d{4}', texto)
            if año_match and 1900 < int(año_match.group()) < 2026:
                año = int(año_match.group())
    except:
        pass

    return {
        "url": url, "plataforma": "pisos",
        "estado_anuncio": "activo",
        "titulo": titulo, "ubicacion": ubicacion, "precio": precio,
        "m2": m2, "habitaciones": habitaciones, "baños": baños,
        "planta": planta, "ascensor": ascensor, "tipo": tipo,
        "estado": estado, "año": año, "anunciante": anunciante,
        "comentario": comentario
    }

# ── LOOP PISOS.COM ───────────────────────────────────
resultados_pisos = []
for i, url in enumerate(listas["pisos"]):
    print(f"\nScraping {i+1}/{len(listas['pisos'])}: {url}")
    try:
        piso = scrape_pisos(driver, url)
        resultados_pisos.append(piso)
        print(f"✓ {piso['estado_anuncio']} — {piso['ubicacion']} — {piso['precio']}€")
    except Exception as e:
        print(f"✗ Error: {e}")
    esperar_entre_requests()

df_pisos = pd.DataFrame(resultados_pisos)
df_pisos.to_csv(f"data/pisos_scraped_{timestamp}.csv", index=False)
print(f"\n✅ Pisos.com — {len(resultados_pisos)} registros")
print(df_pisos["estado_anuncio"].value_counts())


Scraping 1/4: https://www.pisos.com/alquilar/estudio-centro_historico_la_merced29008-60839514551_100500/
✓ alquiler — descartado — None — None€

Scraping 2/4: https://www.pisos.com/alquilar/piso-la_malagueta_monte_sancha-64186596140_109800/
✓ alquiler — descartado — None — None€

Scraping 3/4: https://www.pisos.com/comprar/piso-las_delicias_giron_25_anos_de_paz29003-60052750517_100500/
✓ activo — Calle Alfonso Peña Beeuf — 109000€

Scraping 4/4: https://www.pisos.com/comprar/piso-las_delicias_giron_25_anos_de_paz29003-61745348897_996965/?from_map=true
✓ activo — Calle Alberto Insúa — 190000€

✅ Pisos.com — 4 registros
estado_anuncio
alquiler — descartado    2
activo                   2
Name: count, dtype: int64


### YaEncontre

In [11]:
def scrape_yaencontre(driver, url):
    driver.get(url)
    esperar_carga_pagina()

    soup = BeautifulSoup(driver.page_source, "html.parser")

    texto_pagina = soup.get_text().lower()
    if "anuncio no disponible" in texto_pagina or "ya no está disponible" in texto_pagina:
        return {
            "url": url, "plataforma": "yaencontre",
            "estado_anuncio": "dado de baja",
            "titulo": None, "ubicacion": None, "precio": None,
            "m2": None, "habitaciones": None, "baños": None,
            "planta": None, "ascensor": None, "tipo": None,
            "estado": None, "año": None, "anunciante": None,
            "comentario": None
        }

    try:
        stats_js = driver.execute_script("""
            const header = document.querySelector('.details-header-info');
            return header ? header.innerText : null;
        """)
        stats_text = stats_js.replace('\xa0', ' ')
        stats_clean = stats_text.split("Calcula tu hipoteca")[-1].strip()
    except:
        stats_text = ""
        stats_clean = ""

    try:
        precio_match = re.search(r'([\d\.]+)\s*€', stats_text)
        precio = int(precio_match.group(1).replace(".", "")) if precio_match else None
    except:
        precio = None

    try:
        m2_match = re.search(r'(\d+)\s*m²', stats_clean)
        m2 = int(m2_match.group(1)) if m2_match else None
    except:
        m2 = None

    try:
        numeros = re.findall(r'\b(\d+)\b', stats_clean.split('m²')[0])
        numeros_validos = [int(n) for n in numeros if 0 < int(n) < 20]
        habitaciones = numeros_validos[0] if len(numeros_validos) > 0 else None
        baños = numeros_validos[1] if len(numeros_validos) > 1 else None
    except:
        habitaciones, baños = None, None

    try:
        titulo = soup.find("h1").text.strip()
    except:
        titulo = None

    try:
        ubicacion = titulo.split(" en ")[-1].split(" de ")[0] if titulo and " en " in titulo else titulo
    except:
        ubicacion = titulo

    try:
        features = soup.find_all("li", class_="feature")
        features_dict = {}
        for f in features:
            texto = f.text.strip()
            if ":" in texto:
                key, val = texto.split(":", 1)
                features_dict[key.strip().lower()] = val.strip()
            else:
                features_dict[texto.lower()] = True
    except:
        features_dict = {}

    planta = None
    for key in features_dict:
        if key.startswith("planta"):
            planta = key
            break

    try:
        año_raw = features_dict.get("año de construcción", None)
        año = int(año_raw) if año_raw and año_raw.isdigit() else None
    except:
        año = None

    estado = features_dict.get("estado", None)
    ascensor = "Sí" if "ascensor" in features_dict else "No"
    tipo = "Casa" if titulo and any(k in titulo.lower() for k in [
        "casa", "chalet", "villa", "adosada", "mata", "unifamiliar"
    ]) else "Piso"

    try:
        comentario = soup.find("div", class_="readMoreText").text.strip()
    except:
        comentario = None

    try:
        anunciante = soup.find("div", class_="agency-name").text.strip()
    except:
        anunciante = None

    return {
        "url": url, "plataforma": "yaencontre",
        "estado_anuncio": "activo",
        "titulo": titulo, "ubicacion": ubicacion, "precio": precio,
        "m2": m2, "habitaciones": habitaciones, "baños": baños,
        "planta": planta, "ascensor": ascensor, "tipo": tipo,
        "estado": estado, "año": año, "anunciante": anunciante,
        "comentario": comentario
    }

# ── LOOP YAENCONTRE ──────────────────────────────────
resultados_yaencontre = []
for i, url in enumerate(listas["yaencontre"]):
    print(f"\nScraping {i+1}/{len(listas['yaencontre'])}: {url}")
    try:
        piso = scrape_yaencontre(driver, url)
        resultados_yaencontre.append(piso)
        print(f"✓ {piso['estado_anuncio']} — {piso['ubicacion']} — {piso['precio']}€")
    except Exception as e:
        print(f"✗ Error: {e}")
    esperar_entre_requests()
    if (i + 1) % random.randint(*WAIT_LONG_EVERY) == 0:
        print("Pausa larga...")
        esperar_pausa_larga()

df_yaencontre = pd.DataFrame(resultados_yaencontre)
df_yaencontre.to_csv(f"data/yaencontre_scraped_{timestamp}.csv", index=False)
print(f"\n✅ Yaencontre — {len(resultados_yaencontre)} pisos")
print(df_yaencontre["estado_anuncio"].value_counts())


Scraping 1/12: https://www.yaencontre.com/venta/piso/inmueble-52152-111290263
✓ activo — El Ejido - La Merced - La Victoria — 199000€

Scraping 2/12: https://www.yaencontre.com/venta/piso/inmueble-43586-106237493
✓ activo — El Ejido - La Merced - La Victoria — 225000€

Scraping 3/12: https://www.yaencontre.com/venta/piso/inmueble-66249-111344147
✓ activo — El Molinillo - Capuchinos — 222000€

Scraping 4/12: https://www.yaencontre.com/venta/piso/inmueble-43200-106752978
✓ activo — Conde — None€

Scraping 5/12: https://www.yaencontre.com/venta/piso/inmueble-36105-107761932
✓ activo — Perchel Norte - La Trinidad — 185833€

Scraping 6/12: https://www.yaencontre.com/venta/piso/inmueble-69029-110144529
✓ activo — avenida De Andalucía — 210000€

Scraping 7/12: https://www.yaencontre.com/venta/piso/inmueble-69029-110886907
✓ activo — La Unión - Cruz — None€

Scraping 8/12: https://www.yaencontre.com/venta/piso/inmueble-69029-111321985
✓ activo — Parque Ayala - Jardín — 199000€

Scraping 9/12:

### Tecnocasa

In [12]:
def scrape_tecnocasa(driver, url):
    driver.get(url)
    esperar_carga_pagina()

    soup = BeautifulSoup(driver.page_source, "html.parser")
    texto_completo = driver.execute_script("return document.body.innerText;")

    texto_pagina = soup.get_text().lower()
    if "inmueble no disponible" in texto_pagina or "ya no está disponible" in texto_pagina:
        return {
            "url": url, "plataforma": "tecnocasa",
            "estado_anuncio": "dado de baja",
            "titulo": None, "ubicacion": None, "precio": None,
            "m2": None, "habitaciones": None, "baños": None,
            "planta": None, "ascensor": None, "tipo": None,
            "estado": None, "año": None, "anunciante": "Tecnocasa",
            "comentario": None
        }

    try:
        h1s = driver.execute_script("return Array.from(document.querySelectorAll('h1')).map(h => h.innerText.trim());")
        h1s_validos = [h for h in h1s if "archivo" not in h.lower() and len(h) > 10]
        titulo = h1s_validos[0] if h1s_validos else None
    except:
        titulo = soup.find("h1").text.strip() if soup.find("h1") else None

    try:
        ubicacion = titulo.split(" en ")[-1] if titulo and " en " in titulo else titulo
    except:
        ubicacion = titulo

    try:
        precio_text = driver.execute_script("""
            const all = document.querySelectorAll('*');
            for (let el of all) {
                if (el.children.length === 0 && el.innerText &&
                    el.innerText.includes('€') && el.innerText.length < 20) {
                    return el.innerText.trim();
                }
            }
            return null;
        """)
        precio = int(precio_text.replace(".", "").replace("€", "").strip()) if precio_text else None
    except:
        try:
            precio_text = soup.find("span", class_="current-price").text.strip()
            precio = int(precio_text.replace(".", "").replace("€", "").strip())
        except:
            precio = None

    try:
        m2_match = re.search(r'(\d+)\s*m2\b', texto_completo.lower())
        if not m2_match:
            m2_match = re.search(r'(\d+)\s*m²', driver.title)
        m2 = int(m2_match.group(1)) if m2_match else None
    except:
        m2 = None

    try:
        hab_match = re.search(r'(\d+)\s*dormitor', texto_completo.lower())
        habitaciones = int(hab_match.group(1)) if hab_match else None
    except:
        habitaciones = None

    try:
        ban_match = re.search(r'(\d+)\s*baño', texto_completo.lower())
        baños = int(ban_match.group(1)) if ban_match else None
    except:
        baños = None

    try:
        titulos_f = soup.find_all("div", class_="estate-features-title")
        valores_f = soup.find_all("div", class_="estate-features-value")
        features_dict = {t.text.strip().replace(":", "").lower(): v.text.strip()
                        for t, v in zip(titulos_f, valores_f)}
    except:
        features_dict = {}

    try:
        año_raw = features_dict.get("año de construcción", None)
        if not año_raw:
            año_match = re.search(r'(\d{4})', texto_completo)
            año_raw = año_match.group(1) if año_match and 1900 < int(año_match.group(1)) < 2026 else None
        año = int(año_raw) if año_raw and str(año_raw).isdigit() else None
    except:
        año = None

    try:
        comentario = soup.find("div", class_="estate-description").text.strip()
        comentario = comentario.replace("Descripción del inmueble", "").strip()
        if not comentario:
            idx = texto_completo.find("Descripción")
            comentario = texto_completo[idx:idx+500].strip() if idx > 0 else None
    except:
        comentario = None

    texto_ref = (comentario or "") + texto_completo
    planta_match = re.search(r'(\d+)[ªa]\s*planta', texto_ref.lower())
    planta = f"{planta_match.group(1)}ª planta" if planta_match else None
    ascensor = "Sí" if "ascensor" in texto_ref.lower() else "No"

    estado = None
    if "entrar a vivir" in texto_ref.lower() or "llave en mano" in texto_ref.lower():
        estado = "Para entrar a vivir"
    elif "reformado" in texto_ref.lower():
        estado = "Reformado"
    elif "buen estado" in texto_ref.lower():
        estado = "Buen estado"

    tipo = "Casa" if titulo and any(k in titulo.lower() for k in [
        "casa", "chalet", "villa", "adosada", "mata", "unifamiliar"
    ]) else "Piso"

    return {
        "url": url, "plataforma": "tecnocasa",
        "estado_anuncio": "activo",
        "titulo": titulo, "ubicacion": ubicacion, "precio": precio,
        "m2": m2, "habitaciones": habitaciones, "baños": baños,
        "planta": planta, "ascensor": ascensor, "tipo": tipo,
        "estado": estado, "año": año, "anunciante": "Tecnocasa",
        "comentario": comentario
    }

# ── LOOP TECNOCASA ───────────────────────────────────
resultados_tecnocasa = []
for i, url in enumerate(listas["tecnocasa"]):
    print(f"\nScraping {i+1}/{len(listas['tecnocasa'])}: {url}")
    try:
        piso = scrape_tecnocasa(driver, url)
        resultados_tecnocasa.append(piso)
        print(f"✓ {piso['estado_anuncio']} — {piso['ubicacion']} — {piso['precio']}€")
    except Exception as e:
        print(f"✗ Error: {e}")
    esperar_entre_requests()

df_tecnocasa = pd.DataFrame(resultados_tecnocasa)
df_tecnocasa.to_csv(f"data/tecnocasa_scraped_{timestamp}.csv", index=False)
print(f"\n✅ Tecnocasa — {len(resultados_tecnocasa)} pisos")
print(df_tecnocasa["estado_anuncio"].value_counts())


Scraping 1/2: https://malaga4.tecnocasa.es/malaga/piso-en-venta-643358
✓ activo — C. Carlos Falgueras — 190000€

Scraping 2/2: https://www.tecnocasa.es/venta/piso/malaga/malaga/653845.html
✓ activo — Teatinos - Portada Alta - Carlos Haya — 179900€

✅ Tecnocasa — 2 pisos
estado_anuncio
activo    2
Name: count, dtype: int64


### 4 

In [13]:
driver.quit()
print("✅ Pipeline completo")

✅ Pipeline completo
